### Setup and data ingestion

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
# !PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# !wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
# !wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

### Q1. Generating Questions 

In [4]:
from evaluation_utils import llm_structured_retry
import json 
from pydantic import BaseModel 
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm
import pandas as pd 
import numpy as np 

In [5]:
load_dotenv()
openai_client = OpenAI()

In [6]:
# class that will define the llm output structure 
class Questions(BaseModel): 
    questions: list[str]

In [7]:
doc_q1 = documents[:3]  # for the exercise we work only on the first 3 documents 

In [8]:
# Function that generates questions from the document: 

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

Because we are processing only 3 documents, I will run a simple for loop: 

In [9]:
ground_truth = []
usages = []

for doc in tqdm(doc_q1):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

The generated questions: 

In [10]:
pd.DataFrame(ground_truth)

,question,document
0,"What is Retrieval-Augmented Generation, and wh...",01-agentic-rag/lessons/01-intro.md
1,Why does this lesson treat the language model ...,01-agentic-rag/lessons/01-intro.md
2,What are some common problems with LLMs that t...,01-agentic-rag/lessons/01-intro.md
3,What kind of demo project will be built in thi...,01-agentic-rag/lessons/01-intro.md
4,What topics are covered in the first part of t...,01-agentic-rag/lessons/01-intro.md
5,What do I need installed before starting this ...,01-agentic-rag/lessons/02-environment.md
6,How do I create the project from scratch and w...,01-agentic-rag/lessons/02-environment.md
7,Where should I store my API key so it doesn't ...,01-agentic-rag/lessons/02-environment.md
8,How do I check that the OpenAI client is set u...,01-agentic-rag/lessons/02-environment.md
9,If I want to use Groq or another compatible AP...,01-agentic-rag/lessons/02-environment.md


In [11]:
input_token_usage = np.array([usage.input_tokens for usage in usages])
print(f"The average number of input tokens used across the 3 calls is {input_token_usage.mean():.0f} (~1400)")

The average number of input tokens used across the 3 calls is 1353 (~1400)


### Full Ground Truth & Chunking

In [12]:
# Downloading the ground truth dataset 
# !PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# !wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

In [13]:
ground_truth_df = pd.read_csv("ground-truth.csv")
ground_truth = ground_truth_df.to_dict(orient='records')
ground_truth_df.head(15)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md
5,What do I need installed before starting this ...,01-agentic-rag/lessons/02-environment.md
6,How do I create the new project from scratch w...,01-agentic-rag/lessons/02-environment.md
7,Which packages should I add for the notebooks ...,01-agentic-rag/lessons/02-environment.md
8,What is the recommended way to keep my API key...,01-agentic-rag/lessons/02-environment.md
9,How can I use an OpenAI-compatible provider li...,01-agentic-rag/lessons/02-environment.md


In [14]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

**1. Text Index**

In [15]:
from minsearch import Index 

index = Index(text_fields=['content'], keyword_fields=['filename'])
index.fit(chunks)

def text_search(query, num_results=5): 
    return index.search(query, num_results=num_results)

**2. Vector Search Index** 

In [16]:
!python embed/download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [17]:
from minsearch import VectorSearch
from embed.embedder import Embedder

embedder = Embedder() 
chunks_embeddings = embedder.encode_batch([chunk['content'] for chunk in chunks])
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(chunks_embeddings, chunks)

def vector_search(query, num_results=5): 
    embedding_vec = embedder.encode(query)
    return vindex.search(embedding_vec, num_results=num_results)

2026-06-22 15:51:22.808261469 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


**3. Hybrid Search** 

In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

### Q2. First result with text search

In [19]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [20]:
print(f"The first result of text search is {text_search(q)[0]['filename']}") 

The first result of text search is 01-agentic-rag/lessons/03-rag.md


### Q3. First result with vector search

In [21]:
print(f"The first result of vector search is {vector_search(q)[0]['filename']}") 

The first result of vector search is 01-agentic-rag/lessons/01-intro.md


**Evaluation Metrics** 

In [22]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total


def hit_rate(relevance): 
    cnt = 0 

    for line in relevance: 
        if 1 in line: 
            cnt += 1 

    return cnt / len(relevance)


def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

### Q4. Evaluating Text Search

In [23]:
ts_res = evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [24]:
print(f"The hit rate of text search is {ts_res['hit_rate']:.2f} (mean reciprocal rank is {ts_res['mrr']:.2f})")

The hit rate of text search is 0.76 (mean reciprocal rank is 0.59)


### Q5. Evaluating Vector Search 

In [25]:
vs_res = evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [26]:
print(f"The mean reciprocal rank of text search is {vs_res['mrr']:.2f} (hit rate is {vs_res['hit_rate']:.2f})")

The mean reciprocal rank of text search is 0.55 (hit rate is 0.72)


### Q6. Tuning Hybrid Search

In [27]:
from functools import partial 

In [28]:
k_grid = [1, 50, 100, 200]
tuning_results = []

for k in k_grid: 
    search_func = partial(hybrid_search, k=k)
    eval_res = evaluate(ground_truth, search_func) 

    tuning_results.append(
        {
            'k': k, 
            **eval_res
        }
    )

results_df = pd.DataFrame(tuning_results)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [29]:
results_df.sort_values('mrr', ascending=False)

,k,hit_rate,mrr
0,1,0.838889,0.648194
1,50,0.836111,0.637917
2,100,0.836111,0.637917
3,200,0.836111,0.637917


In [30]:
best_k = results_df.sort_values('mrr', ascending=False).iloc[0]

In [31]:
print(f"The best mrr (mrr={best_k['mrr']:.2f}) is given by k={best_k['k']},", 
      "suggesting that the text and vector retrievers might rank relevant documents already near the top,\n", 
      "while giving weight to lower-ranked candidates appears to be superflous.")

The best mrr (mrr=0.65) is given by k=1.0, suggesting that the text and vector retrievers might rank relevant documents already near the top,
 while giving weight to lower-ranked candidates appears to be superflous.
